# Task 7 — Statevector backend

Walk-through of `Statevector` and `StatevectorSimulator`. Focus on physics-correctness checks that complement the unit tests.

In [ ]:
import numpy as np
from qaravan.backends.statevector import Statevector, StatevectorSimulator, op_action, partial_overlap
from qaravan.core.circuits import Circuit
from qaravan.core.gates import H, X, Z, CNOT, RX, RZ
from qaravan.core.observables import PauliString, PauliSum, LocalOp, Magnetization

## 1. Initialisation paths

Verify each path gives the expected tensor.

In [ ]:
# All-zero state |000⟩
sv = Statevector(3)
print("All-zero tensor[0,0,0]:", sv._tensor[0,0,0])  # should be 1
print("norm:", sv.norm())

# Bitstring
sv_01 = Statevector(bitstring="01")
print("\n|01⟩ via bitstring:")
print("  _tensor[0,1] =", sv_01._tensor[0,1])  # should be 1
print("  _tensor[1,0] =", sv_01._tensor[1,0])  # should be 0

# Haar-random with seed
sv_rnd = Statevector(4, random_seed=42)
print("\nHaar-random 4-qubit norm:", sv_rnd.norm())

## 2. GHZ circuit

**Check by hand:** `to_array()` should be `[1/√2, 0, 0, 1/√2]` for 2 qubits.

In [ ]:
circ = Circuit([H(0), CNOT([0,1])], num_sites=2)
sv0 = Statevector(2)
bell = StatevectorSimulator(circ, sv0).run()

print("Bell state amplitudes:", bell.to_array())
print("Expected:              ", np.array([1,0,0,1])/np.sqrt(2))
assert np.allclose(bell.to_array(), np.array([1,0,0,1])/np.sqrt(2))
print("✓ GHZ / Bell check passed")

## 3. `op_action` correctness — non-contiguous gate

CNOT on qubits 0 and 2 of |100⟩ should give |101⟩.

In [ ]:
CNOT_mat = np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]], dtype=complex)

psi = np.zeros([2,2,2], dtype=complex)
psi[1,0,0] = 1.0  # |100⟩

psi_out = op_action(CNOT_mat, [0,2], psi)

print("After CNOT(0,2) on |100⟩:")
print("  |101⟩ amplitude:", psi_out[1,0,1])  # should be 1
print("  |100⟩ amplitude:", psi_out[1,0,0])  # should be 0
assert np.isclose(psi_out[1,0,1], 1.0)
print("✓ Non-contiguous op_action correct")

## 4. `partial_overlap` — full overlap and RDM

**Full overlap:** `partial_overlap(ψ, ψ, skip=[])` should return `[[1+0j]]` for a normalized state.

**RDM check:** For the Bell state, each qubit's RDM is `I/2`.

In [ ]:
flat = bell.to_array()

# Full overlap
ov = partial_overlap(flat, flat)
print("Full overlap of Bell with itself:", ov)  # [[1+0j]]
assert np.isclose(ov[0,0], 1.0)

# RDM of qubit 0 in Bell = I/2
rdm0 = partial_overlap(flat, flat, skip=[0])
print("\nRDM qubit 0 of Bell state:")
print(rdm0)
assert np.allclose(rdm0, np.eye(2)/2)
print("✓ RDM is maximally mixed for Bell state")

## 5. Expectation values

Physics checks for all three observable types.

In [ ]:
# ⟨Z⟩ on |0⟩ = 1
sv0 = Statevector(1)
print("⟨Z⟩|0⟩ =", sv0.expectation(PauliString("Z")).real)  # 1

# ⟨Z⟩ on |1⟩ = -1
sv1 = Statevector(bitstring="1")
print("⟨Z⟩|1⟩ =", sv1.expectation(PauliString("Z")).real)  # -1

# ⟨X⟩ on |+⟩ = 1
svp = Statevector(array=np.array([1,1])/np.sqrt(2))
print("⟨X⟩|+⟩ =", svp.expectation(PauliString("X")).real)  # 1

# ⟨ZZ⟩ on Bell = 1
print("⟨ZZ⟩ Bell =", bell.expectation(PauliString("ZZ")).real)  # 1

# Magnetization(2) on |01⟩ = 0  (Z_0 = +1, Z_1 = -1, avg = 0)
sv01 = Statevector(bitstring="01")
print("Mag(2) on |01⟩ =", sv01.expectation(Magnetization(2)).real)  # 0

# LocalOp check matches PauliString
Z_mat = np.array([[1,0],[0,-1]], dtype=complex)
exp_pauli = bell.expectation(PauliString("ZZ"))
exp_local = bell.expectation(LocalOp(np.kron(Z_mat, Z_mat), [0,1]))
print("\n⟨ZZ⟩ via PauliString:", exp_pauli.real)
print("⟨ZZ⟩ via LocalOp:    ", exp_local.real)
assert np.isclose(exp_pauli, exp_local)

## 6. `sample` — Born-rule statistics

In [ ]:
shots = bell.sample(10000)
print("Sample shape:", shots.shape)  # (10000, 2)

# Bell state should give only 00 and 11
rows = [tuple(row) for row in shots]
unique = set(rows)
print("Unique outcomes:", unique)   # only (0,0) and (1,1)
assert unique <= {(0,0), (1,1)}

frac_00 = rows.count((0,0)) / 10000
print(f"Fraction |00⟩: {frac_00:.3f}  (expected ~0.5)")
assert abs(frac_00 - 0.5) < 0.02

## 7. `measure_and_collapse` — partial measurement

Measure qubit 0 of the Bell state. The collapsed 2-qubit state should be `|00⟩` or `|11⟩`.

In [ ]:
for trial in range(5):
    sv_post, outcome = bell.measure_and_collapse([0])
    if outcome == "0":
        expected = np.array([1,0,0,0], dtype=complex)
    else:
        expected = np.array([0,0,0,1], dtype=complex)
    ok = np.allclose(sv_post.to_array(), expected)
    print(f"  outcome={outcome}  collapsed=|{outcome}{outcome}⟩  correct={ok}")
    assert ok, f"Collapsed state wrong for outcome '{outcome}'"

## 8. `project_and_renorm` and `reset`

In [ ]:
# project |+⟩ onto '0'
svp = Statevector(array=np.array([1,1])/np.sqrt(2))
sv_proj = svp.project_and_renorm([0], "0")
print("project |+⟩ onto '0':", sv_proj.to_array())  # [1, 0]
assert np.allclose(sv_proj.to_array(), [1, 0])

# reset |1⟩ to |0⟩
sv1 = Statevector(bitstring="1")
sv_reset = sv1.reset([0], reset_to=0)
print("reset |1⟩ → |0⟩:", sv_reset.to_array())  # [1, 0]
assert np.allclose(sv_reset.to_array(), [1, 0])

# rdm of reset qubit 0 in 2-qubit |11⟩ should be |0⟩⟨0|
sv11 = Statevector(bitstring="11")
sv_r = sv11.reset([0], reset_to=0)
print("\nreset qubit 0 of |11⟩ to 0 → state:", sv_r.to_array())  # |01⟩
assert np.allclose(sv_r.to_array(), [0, 1, 0, 0])

## 9. Dynamic circuit pattern

Minimal feed-forward: prepare Bell, measure qubit 0, conditionally apply X to qubit 1.

In [ ]:
results = {"0": 0, "1": 0}

for _ in range(200):
    sv_post, outcome = bell.measure_and_collapse([0])
    if outcome == "1":
        # apply X to qubit 1 to deterministically get |00⟩
        circ_fix = Circuit([X(1)], num_sites=2)
        sv_post = sv_post.apply(circ_fix)
    results[outcome] += 1
    # After correction, state should always be |00⟩
    assert np.allclose(sv_post.to_array(), [1,0,0,0]), f"Correction failed for outcome {outcome}"

print("Outcomes over 200 trials:", results)
print("Post-correction state is always |00⟩ ✓")

## 10. Verify `overlap` and `norm` route through `partial_overlap`

A sanity check that the two primitives cover all inner-product computations.

In [ ]:
sv_rnd1 = Statevector(3, random_seed=10)
sv_rnd2 = Statevector(3, random_seed=11)

# norm via method
print("norm via method:", sv_rnd1.norm())

# overlap via method vs direct
ov_method = sv_rnd1.overlap(sv_rnd2)
ov_direct = np.dot(sv_rnd1.to_array().conj(), sv_rnd2.to_array())
print("overlap via method:", ov_method)
print("overlap via np.dot:", ov_direct)
assert np.isclose(ov_method, ov_direct)
print("✓ overlap consistent")